In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%PandasProfile
### cell 10 ###

gender_map = {
    'Male': 'Man',
    'Female': 'Woman',
    'A different identity': 'Prefer to self-describe',
    'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe',
    'Nonbinary': 'Prefer to self-describe',
    'Prefer not to say': 'Prefer to self-describe'
}


def combine_subset_of_data_from_multiple_years_22(question_of_interest, x_axis_title, include_2017=None):
    # Prepare list of (year, df, column)
    year_dfs = []
    if include_2017 is not None:
        year_dfs.append(('2017', responses_df_2017, 'GenderSelect'))
    year_dfs += [
        ('2022', responses_df_2022_cell10, question_of_interest),
        ('2021', responses_df_2021, question_of_interest),
        ('2020', responses_df_2020, question_of_interest),
        ('2019', responses_df_2019_cell10, question_of_interest),
        ('2018', responses_df_2018_cell10, question_of_interest),
    ]

    frames = []
    for year, df, col in year_dfs:
        # Fast count by original category, then map and aggregate counts only on unique keys
        counts = df[col].value_counts(dropna=False)
        # Consolidate categories via the gender_map on the counts index
        grouped = counts.groupby(
            counts.index.to_series().replace(gender_map),
            dropna=False
        ).sum()
        # Compute percentages
        pct = (grouped.div(grouped.sum()).mul(100).round(1))
        # Build a small DataFrame from the percentages
        tmp = pct.reset_index(name='percentage')
        # Rename the first column (the category) to the x-axis title
        tmp.columns = [x_axis_title, 'percentage']
        tmp['year'] = year
        frames.append(tmp)

    # Combine all years and ensure column order
    result = pd.concat(frames, ignore_index=True)
    return result[['percentage', 'year', x_axis_title]]

# Combine and sort
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''
age_df_combined_cell22 = (
    combine_subset_of_data_from_multiple_years_22(
        question_of_interest_cell22,
        title_for_x_axis_cell22,
        include_2017='yes'
    )
    .sort_values(['year', 'percentage'], ascending=True)
)
age_df_combined_cell22.info()